## NOTE: Script not used..

In [2]:
# --- setup gcam climate
# (we have soil T already, and now need soil moisture and infilt)
import os 

import geopandas as gpd
import numpy as np
import pandas as pd 
import xarray as xr
# [ for optional plotting at bottom of script ]
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.pyplot as plt

# --- read in parquet data
spin_parquet = 's3://carbonplan-carbon-removal/ew-workflows-data/valdation-data/sampled_locations_updated.parquet'
rtgdf = gpd.read_parquet(spin_parquet)
# rtgdf.columns


## Read in daily data for accumulations

In [4]:
# [ get points for sampling ]
rtgdf['lon'] = rtgdf.geometry.x
rtgdf['lat'] = rtgdf.geometry.y

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET    = "era5-land-daily-utc-v1.zarr" # [daily land]
# DATASET    = "reanalysis-era5-land-no-antartica-v0.zarr"  # [hourly land]
TOKEN      = "edh:edh_pat_5006183c5b85f3a21566968ca1710fe2f600be9e71841e2f750adeb4ee818ae35b2595d1d470d641df0560f317133278@data.earthdatahub.destine.eu"
ZARR_URL   = f"https://{TOKEN}/era5/{DATASET}"
TIME_DIM   = "valid_time"   
YEAR_START = 2018
YEAR_END   = 2022   # 5-yr window; widen if RAM allows (~8 GB free recommended)
OUT_FILE   = "era5land_annual_means.nc"

# ERA5-Land variable names — print ds.data_vars below to confirm exact names
VAR_INFILT  = "ro" # "ssro"       # direct variable (m / timestep, accumulated)
VAR_PRECIP  = "tp"           # fallback if above missing (tp)
# VAR_SRO     = "sro"                # fallback (sro)
VAR_SWVL1   = "swvl1" # swvl1 [m³/m³], top 0–7 cm depth

# ── OPEN LAZILY ──────────────────────────────────────────────────────────────
print("Opening zarr store (lazy, no data loaded yet)...")
ds = xr.open_zarr(
    ZARR_URL,
    storage_options={"client_kwargs": {"trust_env": True}},
    chunks={"time": 24, "latitude": 100, "longitude": 100},  # ~1-day time chunks
)
# ── INSPECT ──────────────────────────────────────────────────────────────────
print("\nAvailable variables:")
print(list(ds.data_vars))
print(f"\nFull time range in store: {ds[TIME_DIM].values[0]} → {ds[TIME_DIM].values[-1]}")

Opening zarr store (lazy, no data loaded yet)...


/Users/tylerkukla/Documents/GitHub/ew-workflows/.venv/lib/python3.13/site-packages/zarr/codecs/numcodecs/_codecs.py:161: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)



Available variables:
['d2m', 'e', 'pev', 'ro', 'sp', 'ssr', 'ssrd', 'str', 'swvl1', 'swvl2', 't2m', 'tp', 'u10', 'v10']

Full time range in store: 1950-01-01T00:00:00.000000000 → 2026-03-31T00:00:00.000000000


/var/folders/2r/w7zcr7vx4p5cr7n4d0b51f4c0000gn/T/ipykernel_14381/230067294.py:23: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr(
/var/folders/2r/w7zcr7vx4p5cr7n4d0b51f4c0000gn/T/ipykernel_14381/230067294.py:23: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr(


In [7]:
# ── SUBSET TO TIME WINDOW ─────────────────────────────────────────────────────
print(f"\nSubsetting to {YEAR_START}–{YEAR_END}...")
ds_sub = ds.sel({TIME_DIM: slice(f"{YEAR_START}-01-01", f"{YEAR_END}-12-31")})

# ── SELECT VARS ───────────────────────────────────────────────────────────────
use_direct_infiltration = VAR_INFILT in ds.data_vars
flux_vars = [VAR_INFILT, VAR_SWVL1, VAR_PRECIP]

ds_sub = ds_sub[flux_vars]

# Extract all 32 points at once using nearest-neighbor lookup
ds_points = ds_sub.sel(
    latitude=xr.DataArray(rtgdf.lat.values, dims="points"),
    longitude=xr.DataArray(rtgdf.lon.values, dims="points"),
    method="nearest"
)



Subsetting to 2018–2022...


In [8]:
# ── STEP 1: RESAMPLE TO MONTHLY (keeps memory manageable) ────────────────────
# State variable (m³/m³) → monthly MEAN → later mean to annual
print("\nResampling to monthly aggregates (slow — streaming from remote)...")

# Now daily values are independent so summing is correct
monthly_precip = ds_sub[VAR_PRECIP].resample({TIME_DIM: "1MS"}).sum()
monthly_ssro   = ds_sub[VAR_INFILT].resample({TIME_DIM: "1MS"}).sum()
monthly_swvl1  = ds_sub[VAR_SWVL1].resample({TIME_DIM: "1MS"}).mean()

# ── COMPUTE ───────────────────────────────────────────────────────────────────
print("Computing... (this is the slow step)")
monthly_precip = monthly_precip.compute()
monthly_ssro   = monthly_ssro.compute()
monthly_swvl1  = monthly_swvl1.compute()


Resampling to monthly aggregates (slow — streaming from remote)...
Computing... (this is the slow step)


TimeoutError: 

In [ ]:
# ── STEP 3: INFILTRATION IN MM ────────────────────────────────────────────────
if use_direct_infiltration:
    monthly_infilt_mm = monthly_ssro * 1000.0
else:
    monthly_infilt_mm = (monthly_precip - monthly_sro) * 1000.0  # m → mm
    monthly_infilt_mm = monthly_infilt_mm.clip(min=0)

# ── STEP 4: ANNUAL AGGREGATES ─────────────────────────────────────────────────
annual_infiltration = monthly_infilt_mm.resample({TIME_DIM: "1YS"}).sum()  # mm/yr
annual_swvl1        = monthly_swvl1.resample({TIME_DIM: "1YS"}).mean()     # m³/m³
annual_precip       = monthly_precip.resample({TIME_DIM: "1YS"}).sum()  # m/yr
annual_sro          = monthly_sro.resample({TIME_DIM: "1YS"}).sum()  # m/yr

# ── STEP 5: LONG-TERM MEANS ───────────────────────────────────────────────────
mean_infiltration = annual_infiltration.mean(dim=TIME_DIM)
mean_swvl1        = annual_swvl1.mean(dim=TIME_DIM)
mean_precip       = annual_precip.mean(dim=TIME_DIM)
mean_sro          = annual_sro.mean(dim=TIME_DIM)



# ── STEP 6: SAVE OUTPUT ───────────────────────────────────────────────────────
infilt_method = "ERA5-Land soil_water_infiltration" if use_direct_infiltration \
                else "total_precipitation minus surface_runoff"

ds_out = xr.Dataset({
    # Per-year maps
    "annual_infiltration_mmyr": annual_infiltration.rename("annual_infiltration_mmyr"),
    "annual_soil_moisture_vol": annual_swvl1.rename("annual_soil_moisture_vol"),
    # Long-term mean maps
    "mean_infiltration_mmyr":   mean_infiltration.rename("mean_infiltration_mmyr"),
    "mean_soil_moisture_vol":   mean_swvl1.rename("mean_soil_moisture_vol"),
})

ds_out["mean_infiltration_mmyr"].attrs = {
    "long_name":  f"Mean annual soil water infiltration ({YEAR_START}-{YEAR_END})",
    "units":      "mm yr-1",
    "source":     infilt_method,
}
ds_out["mean_soil_moisture_vol"].attrs = {
    "long_name":  f"Mean volumetric soil water content, layer 1 0-7 cm ({YEAR_START}-{YEAR_END})",
    "units":      "m3 m-3",
    "note":       (
        "This is volumetric water content (theta). "
        "To approximate void ratio: e = theta / (1 - phi), "
        "where phi (porosity) is ~0.40-0.55 depending on soil texture. "
        "Porosity is not available in ERA5-Land directly — use a soil texture "
        "dataset (e.g. SoilGrids) for spatially explicit porosity."
    ),
}
ds_out.attrs = {
    "source":      "ERA5-Land via DestinE Earth Data Hub",
    "years":       f"{YEAR_START}-{YEAR_END}",
    "created_by":  "era5land_infiltration_soilmoisture.py",
}

# ds_out.to_netcdf(OUT_FILE)
# print(f"\nSaved → {OUT_FILE}")
ds_out